# Day 2: Calculus Essentials — Gradients & Chain Rule

**Module 1 — Foundations | 100 Days of Data Science**

## Why This Matters
Neural networks *learn* by adjusting weights to reduce error. That adjustment process — backpropagation — is nothing but the **chain rule** from calculus applied over and over, layer by layer.

If Day 1 was "how a neuron computes its output," Day 2 is "how a neuron learns from its mistakes."

## Topics Covered Today
1. Derivatives — the core idea
2. Partial derivatives & gradients
3. The chain rule
4. Gradient descent from scratch
5. Autograd: how PyTorch does this automatically
6. Practice exercises

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("NumPy version:", np.__version__)

---
## 1. Derivatives — The Core Idea

A derivative tells you **how fast a function's output changes** as you nudge its input slightly.

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

In deep learning: if `f` is the loss function and `x` is a weight, the derivative tells you **which direction to move the weight to reduce loss**.

In [ ]:
def f(x):
    return x**2

def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

x = 3
print(f"f(x) = x^2, at x={x}")
print("Numerical derivative:", numerical_derivative(f, x))
print("Analytical derivative (2x):", 2 * x)

### Visualizing a Derivative as Slope

In [ ]:
xs = np.linspace(-5, 5, 100)
ys = xs**2

x0 = 2
slope = 2 * x0
tangent = slope * (xs - x0) + x0**2

plt.figure(figsize=(6,4))
plt.plot(xs, ys, label='f(x) = x^2')
plt.plot(xs, tangent, '--', label=f'tangent line at x={x0} (slope={slope})')
plt.scatter([x0], [x0**2], color='red')
plt.legend()
plt.title('The derivative is the slope of the tangent line')
plt.grid(True)
plt.show()

---
## 2. Partial Derivatives & Gradients

When a function has multiple inputs (like a neural network with many weights), we take a **partial derivative** with respect to each input separately, holding the others constant.

The **gradient** is just the vector of all partial derivatives — it points in the direction of steepest increase.

$$\nabla f = \left[\frac{\partial f}{\partial x}, \frac{\partial f}{\partial y}\right]$$

In [ ]:
# f(x, y) = x^2 + y^2
def f2(x, y):
    return x**2 + y**2

def gradient_f2(x, y, h=1e-5):
    df_dx = (f2(x + h, y) - f2(x - h, y)) / (2 * h)
    df_dy = (f2(x, y + h) - f2(x, y - h)) / (2 * h)
    return np.array([df_dx, df_dy])

x, y = 3, 4
grad = gradient_f2(x, y)
print(f"Gradient at ({x},{y}):", grad)
print("Analytical gradient (2x, 2y):", (2*x, 2*y))

---
## 3. The Chain Rule

If `y = f(g(x))`, then:

$$\frac{dy}{dx} = \frac{dy}{dg} \cdot \frac{dg}{dx}$$

This is exactly how backpropagation works: error is passed **backward through layers**, multiplying local gradients at each step.

In [ ]:
# Example: y = (3x + 1)^2
# Let g(x) = 3x + 1, f(g) = g^2
# dy/dg = 2g, dg/dx = 3
# dy/dx = 2g * 3 = 6g = 6(3x+1)

def g(x):
    return 3*x + 1

def y_func(x):
    return g(x)**2

x = 2
numerical = numerical_derivative(y_func, x)
analytical = 6 * (3*x + 1)

print("Numerical dy/dx:", numerical)
print("Analytical dy/dx (chain rule):", analytical)

### Mini Neuron Example (Chain Rule in Action)
A tiny 2-step computation: `z = w*x + b`, `output = sigmoid(z)`. We compute how `output` changes with respect to `w` — this is literally one step of backpropagation.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

x, w, b = 1.5, 0.8, 0.1
z = w * x + b
output = sigmoid(z)

# Chain rule: d(output)/dw = d(output)/dz * dz/dw
d_output_d_z = sigmoid_derivative(z)
dz_dw = x  # since z = w*x + b, dz/dw = x
d_output_dw = d_output_d_z * dz_dw

print("z =", z)
print("output =", output)
print("gradient of output w.r.t w:", d_output_dw)

---
## 4. Gradient Descent From Scratch

Now we use the gradient to actually **minimize** a function — this is exactly how model weights are updated during training.

$$w_{new} = w_{old} - \text{learning\_rate} \times \frac{\partial Loss}{\partial w}$$

In [ ]:
# Minimize f(x) = (x - 3)^2  -> minimum at x = 3
def loss(x):
    return (x - 3)**2

def loss_grad(x):
    return 2 * (x - 3)

x = 0.0  # random starting point
lr = 0.1
history = [x]

for step in range(30):
    grad = loss_grad(x)
    x = x - lr * grad
    history.append(x)

print("Final x (should approach 3):", x)

plt.figure(figsize=(6,4))
plt.plot(history, marker='o')
plt.axhline(3, color='red', linestyle='--', label='true minimum')
plt.xlabel('Step')
plt.ylabel('x value')
plt.title('Gradient Descent Converging to Minimum')
plt.legend()
plt.grid(True)
plt.show()

---
## 5. Autograd: How PyTorch Does This Automatically

In practice, you never compute gradients by hand — frameworks like PyTorch build a computation graph and calculate all gradients automatically via `.backward()`. Here's the same mini-neuron example from Section 3, done with autograd.

*(Requires `pip install torch` to run this cell.)*

In [ ]:
try:
    import torch

    x = torch.tensor(1.5)
    w = torch.tensor(0.8, requires_grad=True)
    b = torch.tensor(0.1, requires_grad=True)

    z = w * x + b
    output = torch.sigmoid(z)

    output.backward()  # computes all gradients automatically

    print("output:", output.item())
    print("gradient of output w.r.t w (autograd):", w.grad.item())
    print("Compare to our manual calculation above — should match closely.")
except ImportError:
    print("PyTorch not installed. Run: pip install torch")

---
## 6. Practice Exercises
Try these before Day 3:

1. Compute the numerical and analytical derivative of `f(x) = 3x^3 - 2x` at `x = 2`.
2. Compute the gradient of `f(x, y) = x^2 * y + y^3` at `(1, 2)`.
3. Use the chain rule to find `dy/dx` for `y = sin(2x + 1)` at `x = 0` (numerically vs analytically).
4. Modify the gradient descent example to minimize `f(x) = (x + 5)^2` instead, and plot convergence.
5. In your own words: why do we use the *negative* gradient in gradient descent, not the positive one?

In [ ]:
# Your practice code here


---
## Summary
- **Derivative** = rate of change = slope of a function at a point
- **Gradient** = vector of partial derivatives, points toward steepest increase
- **Chain rule** = how gradients flow backward through layered computations (backpropagation)
- **Gradient descent** = repeatedly stepping opposite the gradient to minimize loss
- **Autograd** (PyTorch) = automates all of this so you never hand-derive gradients in practice

Next up: **Day 3 — Probability & Information Theory (entropy, cross-entropy)**

---
*Part of the 100 Days of Data Science series | DL-for-Data-Science repo*